<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/9_Databricks_Serverless_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⚠️ **Plataforma recomendada: Databricks Free/Community 2026 con compute serverless. El notebook evita DBFS legacy y prioriza tablas, Volumes y Spark SQL.**

# Databricks: tutorial completo de introduccion

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos -- Big Data

![Universidad Central](https://www.ucentral.edu.co/themes/ucentral/img/template/Universidad%20Central.png)

> **Sesión 9** · 2026

## Proposito pedagogico

Esta sesion es una **primera introduccion guiada a Databricks Free Edition / Community 2026** despues
de haber estudiado Hadoop, YARN y Spark en la sesion anterior. La meta no es
memorizar comandos aislados: la meta es entender donde viven los datos, como se
ejecuta Spark dentro de Databricks y como se construye un flujo reproducible.

## Alcance de la sesion

Trabajaremos con Databricks en su edicion gratuita 2026, que usa computo
serverless y Unity Catalog. Por eso evitaremos patrones legacy como depender de
DBFS root o de `sparkContext`, y usaremos Spark SQL, PySpark DataFrames, tablas,
Volumes cuando esten disponibles, Parquet y Delta Lake.

## Agenda sugerida

1. Entender la interfaz de Databricks gratuito/serverless.
2. Aprender comandos magicos, `dbutils`, Unity Catalog y Volumes.
3. Leer, transformar y escribir datos con CSV, JSON, Parquet y Delta.
4. Comprender Spark: schemas, SQL, funciones, lazy evaluation y planes.
5. Comparar Spark con Pandas y Dask.
6. Introducir Delta Lake, Lakeflow y Workflows.
7. Cerrar con un taller aplicado.

## Por que importa

Databricks permite pasar de un notebook exploratorio a una plataforma de datos:
tablas gobernadas, permisos, lineage, ejecuciones programadas, optimizacion y
pipelines. Ese cambio es central en Big Data moderno.

## Correspondencia con la sesion anterior

| Sesion 7 | En esta sesion |
|---|---|
| Hadoop y YARN explican la administracion de recursos | Databricks serverless abstrae gran parte de esa administracion |
| Spark como motor distribuido | Spark se usa con SQL, DataFrames y PySpark |
| Clusters y ejecucion distribuida | SparkSession, Spark Connect, Jobs, Stages y Tasks |
| Archivos y almacenamiento | Unity Catalog, Volumes y tablas administradas |

Conservamos la intuicion distribuida de la sesion 7, pero la llevamos al flujo
actual de Databricks.

## Contenido

- 0. Databricks Free/Community 2026: serverless y plataforma moderna
- 1. Magic commands y dbutils
- 2. SparkSession y Spark Connect
- 3. Catalogos, tablas y Volumes
- 4. Spark SQL completo: TempViews, DDL y DML
- 5. Tipos de datos y schemas
- 6. Lectura y escritura: CSV, JSON, Parquet y Delta
- 7. Lazy evaluation, Catalyst, Jobs, Stages y repartition
- 8. Photon y Liquid Clustering
- 9. Funciones de cadenas, fechas y colecciones
- 10. Transformaciones completas de la API PySpark
- 11. Por que Spark sobre Pandas, y cuando no
- 12. Por que Spark sobre Dask, y cuando no
- 13. Delta Lake avanzado
- 14. Lakeflow / Delta Live Tables
- 15. Databricks Workflows y Jobs
- 16. Taller end-to-end

---
# Sección 0 -- Databricks Free/Community 2026: serverless y plataforma moderna

## Definicion formal

**Databricks Free Edition** es la version gratuita actual de Databricks para
estudiantes, docentes y personas que estan aprendiendo. En 2026 reemplaza a la
antigua Community Edition y funciona en un entorno **serverless**, con cuotas y
algunas limitaciones.

## Intuicion

En este entorno no administramos nodos manualmente. El estudiante abre un
notebook y Databricks conecta compute serverless. Esto hace mas simple la clase,
pero exige usar patrones modernos: DataFrames, Spark SQL, Unity Catalog, tablas
y Volumes cuando esten disponibles.

| Aspecto | Databricks Free/Community 2026 |
|---|---|
| Compute | Serverless administrado |
| Infraestructura | No se eligen nodos manualmente |
| `sparkContext` | Puede no estar disponible por Spark Connect |
| Archivos | Preferir Volumes, tablas o archivos del workspace |
| DBFS root / FileStore | Legacy o acceso limitado |
| Observabilidad | Query Profile / query insights |

## Ecosistema actual

Databricks hoy no es solo "Spark en la nube". Incluye notebooks, SQL, Workflows,
Unity Catalog, Volumes, Delta Lake, Photon, Liquid Clustering, Lakeflow, Model
Serving y herramientas AI/BI como Genie. En la edicion gratuita pueden existir
cuotas o funciones limitadas, pero el modelo mental moderno es el mismo.

In [ ]:
# Deteccion inicial del entorno Databricks
import sys

print(f"Python: {sys.version}")
print(f"Spark : {spark.version}")

IS_SERVERLESS = False
HAS_SPARK_CONTEXT = False
HAS_UNITY_CATALOG = False

try:
    print("sparkContext.master:", spark.sparkContext.master)
    HAS_SPARK_CONTEXT = True
except Exception as exc:
    IS_SERVERLESS = True
    print("sparkContext no disponible directamente. Probable Spark Connect / Serverless.")
    print(f"Detalle: {type(exc).__name__}: {exc}")

try:
    current_cat = "hive_metastore"
    current_schema = "default"
    current_cat = spark.sql("SELECT current_catalog()").first()[0]
    current_schema = spark.sql("SELECT current_schema()").first()[0]
    HAS_UNITY_CATALOG = current_cat not in ("", None, "hive_metastore")
    print(f"Catalogo actual: {current_cat}")
    print(f"Schema actual  : {current_schema}")
    print(f"Unity Catalog : {HAS_UNITY_CATALOG}")
except Exception as exc:
    print(f"No fue posible detectar catalogo: {exc}")

try:
    photon = spark.conf.get("spark.databricks.photon.enabled", "false")
except Exception:
    photon = "no detectable"

print(f"IS_SERVERLESS={IS_SERVERLESS}, UC={HAS_UNITY_CATALOG}, Photon={photon}")

def nombre_tabla(nombre):
    if HAS_UNITY_CATALOG:
        return f"{current_cat}.{current_schema}.{nombre}"
    return f"{current_schema}.{nombre}"

### Como interpretar el resultado -- deteccion del entorno

- En Databricks Free/Community 2026 es normal que `sparkContext` no este disponible directamente.
- Si hay Unity Catalog, conviene trabajar con tablas y Volumes gobernados.
- La funcion `nombre_tabla` permite usar dos o tres niveles segun el entorno.

In [ ]:
# Instalar dependencias de apoyo
# Regla Databricks moderna: usar %pip, no %sh pip.
%pip install "dask[dataframe]>=2024.1" pyarrow -q

## Advertencia comun

`%sh pip install paquete` instala en el entorno del sistema operativo de la sesion,
pero el interprete Python del notebook puede no ver esa instalacion. `%pip`
instala en el entorno activo del notebook y es el patron recomendado.

In [ ]:
# Pregunta interactiva 1 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 1 de 14 -- Databricks</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> En esta clase se trabaja con Databricks Free/Community 2026 sobre serverless.
  </div>
  <p><strong>Que idea describe mejor este entorno?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q1" value="A"> A. Spark desaparece</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q1" value="B"> B. Permite practicar notebooks, Spark SQL y PySpark con recursos serverless</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q1" value="C"> C. Solo se puede usar Pandas</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q1" value="D"> D. No existen tablas</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q1]:checked');
    var out = document.getElementById('fb_q1');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'B') {
      out.innerHTML = 'Correcto. Databricks gratuito/serverless es suficiente para aprender el flujo base de Spark y tablas.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Databricks gratuito/serverless es suficiente para aprender el flujo base de Spark y tablas.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q1" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 1 -- Magic commands y dbutils

## Definicion formal

Los **magic commands** son comandos especiales de notebook que cambian el modo de
ejecucion de una celda. `dbutils` es una utilidad propia de Databricks para
interactuar con archivos, widgets, secretos y ejecuciones de notebooks.

| Magic | Uso |
|---|---|
| `%python` | Ejecutar Python |
| `%sql` | Ejecutar SQL |
| `%md` | Escribir Markdown |
| `%pip` | Instalar librerias en el entorno del notebook |
| `%run` | Incluir otro notebook |
| `%fs` | Comandos de archivos Databricks |
| `%sh` | Shell del entorno; puede estar limitado en serverless |

## Intuicion

Un notebook puede combinar explicacion, SQL y Python. Para una primera clase,
conviene aprender el equivalente Python de casi todo, porque permite copiar el
codigo dentro de funciones o jobs.

In [ ]:
# SQL desde Python: equivalente portable a una celda %sql
consulta = spark.sql('''
SELECT
  current_catalog() AS catalogo,
  current_schema()  AS schema,
  current_date()    AS fecha_actual
''')
consulta.show(truncate=False)

### Como interpretar el resultado -- magic SQL desde Python

- La salida confirma el catalogo y schema activos.
- `spark.sql` permite usar SQL multi-linea dentro de una celda Python.
- Esto sera util cuando necesitemos DDL, DML o consultas con CTEs.

## Modulos frecuentes de `dbutils`

| Modulo | Para que sirve |
|---|---|
| `dbutils.fs` | Listar y manipular archivos accesibles por Databricks |
| `dbutils.widgets` | Parametrizar notebooks |
| `dbutils.secrets` | Leer credenciales almacenadas en secret scopes |
| `dbutils.notebook` | Ejecutar o terminar notebooks desde codigo |

En Databricks Free/Community 2026 el compute es serverless. El acceso a DBFS
root o FileStore puede estar limitado, por eso el patron recomendado es:
tablas, Volumes de Unity Catalog o archivos del workspace.

In [ ]:
# Explorar ubicaciones de forma segura en Databricks serverless
# Evitamos listar dbfs:/FileStore o /Volumes/ directamente porque pueden fallar
# por permisos o por las limitaciones del compute serverless.

print("Catalogo y schema actuales:")
spark.sql("SELECT current_catalog() AS catalogo, current_schema() AS schema").show(truncate=False)

print("Tablas visibles en el schema actual:")
spark.sql("SHOW TABLES").show(truncate=False)

try:
    catalogo_actual = spark.sql("SELECT current_catalog()").first()[0]
    schema_actual = spark.sql("SELECT current_schema()").first()[0]
    print(f"Volumes disponibles en {catalogo_actual}.{schema_actual}:")
    spark.sql(f"SHOW VOLUMES IN {catalogo_actual}.{schema_actual}").show(truncate=False)
except Exception as exc:
    print("No se pudieron listar Volumes en este schema.")
    print("Esto puede pasar si no hay Volumes creados o si faltan permisos.")
    print(f"Detalle: {type(exc).__name__}: {exc}")

print("\nPatrones recomendados:")
print("- Tabla administrada: catalog.schema.mi_tabla")
print("- Volume si existe: /Volumes/<catalog>/<schema>/<volume>/<archivo>")
print("- Archivo del workspace para ejemplos pequenos")

In [ ]:
# Widgets: parametros simples para notebooks y jobs
dbutils.widgets.text("catalogo_param", "samples", "Catalogo")
dbutils.widgets.dropdown("modo_ejecucion", "demo", ["demo", "produccion"], "Modo")

catalogo_param = dbutils.widgets.get("catalogo_param")
modo_ejecucion = dbutils.widgets.get("modo_ejecucion")

print(f"catalogo_param={catalogo_param}")
print(f"modo_ejecucion={modo_ejecucion}")

In [ ]:
# Secrets y ejecucion de notebooks: patrones seguros
try:
    scopes = dbutils.secrets.listScopes()
    print("Secret scopes disponibles:")
    for s in scopes:
        print(" ", s.name)
except Exception as exc:
    print(f"No fue posible listar secret scopes: {exc}")

print("\nPatron correcto para credenciales:")
print("token = dbutils.secrets.get(scope='mi_scope', key='mi_token')")
print("\nPatron para invocar otro notebook desde un workflow:")
print("dbutils.notebook.run('/Repos/proyecto/otro_notebook', 300, {'fecha': '2026-01-01'})")

In [ ]:
# Pregunta interactiva 2 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 2 de 14 -- dbutils</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Los notebooks se parametrizan para jobs.
  </div>
  <p><strong>Que modulo permite crear parametros visibles en el notebook?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q2" value="A"> A. dbutils.widgets</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q2" value="B"> B. dbutils.fs</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q2" value="C"> C. dbutils.secrets</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q2" value="D"> D. spark.catalog</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q2]:checked');
    var out = document.getElementById('fb_q2');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'A') {
      out.innerHTML = 'Correcto. `dbutils.widgets` crea parametros de entrada.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. `dbutils.widgets` crea parametros de entrada.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q2" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 2 -- SparkSession y Spark Connect

## Definicion formal

**SparkSession** es la puerta principal para usar Spark desde PySpark.
**Spark Connect** es el modelo cliente-servidor usado por el compute serverless:
el notebook envia planes al servidor Spark, y Spark los analiza, optimiza y
ejecuta.

## Intuicion

En Databricks Free/Community 2026 no conviene depender de `sparkContext` ni de
RDDs. Para aprender bien Databricks, usa `SparkSession`, DataFrames y SQL.

| API | Recomendacion para esta clase |
|---|---|
| `spark.sql(...)` | Usar |
| `spark.read.table(...)` | Usar |
| DataFrame API | Usar |
| `spark.sparkContext.parallelize(...)` | Evitar en serverless |
| RDDs | No usarlos como patron de clase |
| Global temp views | Evitar en clase; usar temp views o tablas |

In [ ]:
# Lo que funciona bien: SparkSession, SQL y DataFrames
from pyspark.sql import functions as F

df = spark.range(10).withColumn("cuadrado", F.col("id") * F.col("id"))
df.show()

spark.sql("SELECT 1 + 1 AS suma").show()

# Alternativa moderna a sparkContext.parallelize(...)
df_local = spark.createDataFrame([(1,), (2,), (3,)], ["valor"])
df_local.show()

### Como interpretar el resultado -- SparkSession y SparkContext

- El ejemplo muestra tres patrones compatibles: `spark.range`, `spark.sql` y `spark.createDataFrame`.
- Para una introduccion, basta pensar que Python describe un plan y Spark lo ejecuta en el cluster.
- Aunque algunos ejemplos antiguos usen RDDs, los DataFrames son el patron central del curso.

In [ ]:
# Pregunta interactiva 3 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 3 de 14 -- Spark</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> En serverless se usa Spark Connect y no conviene depender de RDDs.
  </div>
  <p><strong>Que API conviene priorizar?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q3" value="A"> A. RDDs</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q3" value="B"> B. sparkContext.parallelize</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q3" value="C"> C. DataFrames y Spark SQL</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q3" value="D"> D. Loops locales con collect</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q3]:checked');
    var out = document.getElementById('fb_q3');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'C') {
      out.innerHTML = 'Correcto. DataFrames y SQL son el patron principal y optimizable.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. DataFrames y SQL son el patron principal y optimizable.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q3" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 3 -- Catalogos, tablas y Volumes

## Definicion formal

Databricks organiza tablas en catalogos y schemas. En Databricks Free/Community
2026 se trabaja con Unity Catalog y nombres como `catalog.schema.table` cuando
estan disponibles. Los **Volumes** son el lugar recomendado para archivos no
tabulares, como CSV, JSON, Parquet suelto o imagenes.

```
catalog
  schema
    table | view | function | volume
```

## Intuicion

La pregunta correcta es: "en que catalogo, schema, tabla o Volume vive el dato".
No asumimos que `dbfs:/FileStore` existe o que se puede listar desde serverless.

In [ ]:
# Explorar catalogos, schema y tablas de ejemplo
spark.sql("SHOW CATALOGS").show(truncate=False)

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = spark.sql("SELECT current_schema()").first()[0]
print(f"Catalogo activo: {CATALOG}")
print(f"Schema activo  : {SCHEMA}")

spark.sql("SHOW TABLES IN samples.nyctaxi").show(truncate=False)

In [ ]:
# Leer y validar el schema real de samples.nyctaxi.trips
from pyspark.sql import functions as F

TAXI_TABLE = "samples.nyctaxi.trips"
sdf = spark.read.table(TAXI_TABLE)

columnas_esperadas = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "pickup_zip",
    "dropoff_zip",
]
columnas_reales = sdf.columns
columnas_faltantes = [c for c in columnas_esperadas if c not in columnas_reales]

print(f"Tabla: {TAXI_TABLE}")
print(f"Columnas reales: {columnas_reales}")

if columnas_faltantes:
    raise ValueError(f"Faltan columnas esperadas en {TAXI_TABLE}: {columnas_faltantes}")

sdf = sdf.select(*columnas_esperadas)

sdf.createOrReplaceTempView("taxi_source_v")

def leer_taxi():
    return spark.table("taxi_source_v")

print(f"Filas: {sdf.count():,}")
print(f"Columnas: {len(sdf.columns)}")
sdf.printSchema()

print("Metadatos del catalogo:")
spark.sql(f"DESCRIBE TABLE {TAXI_TABLE}").show(truncate=False)

### Como interpretar el resultado -- tabla de muestra

- La tabla oficial `samples.nyctaxi.trips` tiene 6 columnas en Databricks Free/Community 2026.
- El schema nos dice tipos de columnas antes de transformar datos.
- Todas las transformaciones posteriores se basan solo en esas columnas verificadas.

In [ ]:
# Volumes y rutas modernas
try:
    spark.sql("SHOW VOLUMES IN samples.nyctaxi").show(truncate=False)
except Exception as exc:
    print("No hay Volumes visibles en samples.nyctaxi o faltan permisos.")
    print(f"Detalle: {exc}")

print("Ruta de Volume en Databricks:")
print("/Volumes/<catalog>/<schema>/<volume>/<archivo>")
print("Ejemplo: /Volumes/main/bronze/raw_files/ventas.parquet")

## Error comun

No uses `C:\Users\estudiante\Downloads\archivo.csv` dentro de Databricks.
Esa ruta existe en el computador local, no en el compute de Databricks. Primero
sube el archivo a un Volume, a archivos del workspace o crea una tabla.

In [ ]:
# Pregunta interactiva 4 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 4 de 14 -- Tablas</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Databricks serverless no ve directamente el disco local del estudiante.
  </div>
  <p><strong>Cual ruta NO debe usarse dentro de Databricks para leer datos del computador local?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q4" value="A"> A. catalog.schema.table</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q4" value="B"> B. /Volumes/catalog/schema/volume/datos.csv</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q4" value="C"> C. archivo subido al workspace</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q4" value="D"> D. C:/datos/trips.csv</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q4]:checked');
    var out = document.getElementById('fb_q4');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'D') {
      out.innerHTML = 'Correcto. Databricks no ve directamente el disco local; se debe subir el archivo o usar Volumes/tablas.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Databricks no ve directamente el disco local; se debe subir el archivo o usar Volumes/tablas.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q4" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 4 -- Spark SQL completo: TempViews, DDL y DML

## Definicion formal

**Spark SQL** permite consultar DataFrames y tablas usando SQL. Una **TempView**
es una vista temporal de sesion creada desde un DataFrame. **DDL** crea o modifica
objetos; **DML** inserta, actualiza o elimina datos.

| Concepto | Ejemplo |
|---|---|
| TempView | `df.createOrReplaceTempView('v')` |
| DDL | `CREATE TABLE`, `DROP TABLE`, `DESCRIBE TABLE` |
| DML | `INSERT INTO`, `MERGE`, `DELETE`, `UPDATE` |

Para compartir resultados entre sesiones, prefiere tablas en el metastore
(`catalog.schema.mi_tabla` cuando Unity Catalog esta disponible). Si tu workspace
usa un metastore legacy, el notebook ajusta el nombre con la funcion `nombre_tabla`.

In [ ]:
# Crear TempView desde un DataFrame y consultarla con SQL
taxi_sample = (
    leer_taxi()
    .select("tpep_pickup_datetime", "fare_amount", "trip_distance", "pickup_zip", "dropoff_zip")
    .where("fare_amount > 0 AND trip_distance > 0")
    .withColumn("tarifa_por_milla", F.col("fare_amount") / F.col("trip_distance"))
    .limit(10000)
)

taxi_sample.createOrReplaceTempView("taxi_sample_v")

spark.sql('''
SELECT
  COUNT(*) AS viajes,
  ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
  ROUND(AVG(tarifa_por_milla), 2) AS tarifa_por_milla_promedio
FROM taxi_sample_v
''').show()

### Como interpretar el resultado -- TempView

- La vista temporal no crea una tabla permanente.
- Permite mezclar PySpark y SQL sin duplicar datos.
- Desaparece al terminar la sesion del notebook.

In [ ]:
# DDL: crear, describir y eliminar una tabla de practica
SQL_TABLE = nombre_tabla("sesion9_sql_demo")

spark.sql(f"DROP TABLE IF EXISTS {SQL_TABLE}")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SQL_TABLE} (
  id BIGINT,
  ciudad STRING,
  valor DOUBLE
)
USING DELTA
''')

if HAS_UNITY_CATALOG:
    spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)
else:
    spark.sql(f"SHOW TABLES IN {SCHEMA}").show(truncate=False)
spark.sql(f"DESCRIBE TABLE {SQL_TABLE}").show(truncate=False)

In [ ]:
# DML: INSERT INTO y consultas de verificacion
spark.sql(f'''
INSERT INTO {SQL_TABLE} VALUES
  (1, 'Bogota', 120.5),
  (2, 'Cali', 95.0),
  (3, 'Medellin', 150.0)
''')

spark.sql(f"SELECT * FROM {SQL_TABLE} ORDER BY id").show()
spark.sql(f"SHOW COLUMNS IN {SQL_TABLE}").show(truncate=False)
spark.sql(f"SHOW CREATE TABLE {SQL_TABLE}").show(truncate=False)

In [ ]:
# CTEs: consultas legibles en varios pasos
spark.sql(f'''
WITH base AS (
  SELECT ciudad, valor
  FROM {SQL_TABLE}
  WHERE valor > 0
),
resumen AS (
  SELECT ciudad, COUNT(*) AS n, ROUND(AVG(valor), 2) AS promedio
  FROM base
  GROUP BY ciudad
)
SELECT *
FROM resumen
ORDER BY promedio DESC
''').show()

In [ ]:
# Pregunta interactiva 5 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 5 de 14 -- Spark SQL</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Una TempView vive durante la sesion.
  </div>
  <p><strong>Que conviene usar para persistir resultados?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q5" value="A"> A. TempView</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q5" value="B"> B. Tabla administrada</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q5" value="C"> C. Variable Python</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q5" value="D"> D. print</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q5]:checked');
    var out = document.getElementById('fb_q5');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'B') {
      out.innerHTML = 'Correcto. Una tabla administrada permanece disponible despues de la celda.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Una tabla administrada permanece disponible despues de la celda.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q5" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 5 -- Tipos de datos y schemas

## Definicion formal

Un **schema** describe las columnas de un DataFrame: nombre, tipo y nulabilidad.
Spark puede inferirlo, pero en pipelines reales conviene declararlo.

| Tipo | Uso |
|---|---|
| `IntegerType`, `LongType` | Enteros |
| `DoubleType` | Numeros decimales |
| `StringType` | Texto |
| `BooleanType` | Verdadero/falso |
| `DateType`, `TimestampType` | Fechas y tiempos |
| `ArrayType`, `MapType`, `StructType` | Datos semiestructurados |

## Intuicion

El schema es el contrato del dato. Si el contrato cambia sin control, los
resultados dejan de ser confiables.

In [ ]:
# Schema explicito con StructType
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType
)
from pyspark.sql import functions as F

schema_ventas = StructType([
    StructField("ciudad", StringType(), False),
    StructField("categoria", StringType(), True),
    StructField("valor", DoubleType(), True),
    StructField("unidades", IntegerType(), True),
    StructField("fecha_txt", StringType(), True),
])

datos_ventas = [
    ("Bogota", "tecnologia", 1200000.0, 2, "2026-01-05"),
    ("Cali", "hogar", 380000.0, 1, "2026-01-06"),
    ("Medellin", "salud", 210000.0, 3, "2026-01-07"),
]

ventas = spark.createDataFrame(datos_ventas, schema_ventas)
ventas.printSchema()
print(ventas.schema)
print(ventas.dtypes)
ventas.show()

In [ ]:
# Conversiones con cast, to_date, to_timestamp y try_cast en SQL
ventas_cast = (
    ventas
    .withColumn("valor_int", F.col("valor").cast("long"))
    .withColumn("fecha", F.to_date("fecha_txt"))
    .withColumn("fecha_ts", F.to_timestamp("fecha_txt"))
)
ventas_cast.show()

ventas_cast.createOrReplaceTempView("ventas_cast_v")
spark.sql('''
SELECT
  ciudad,
  valor,
  try_cast(valor AS INT) AS valor_try_int,
  try_cast('texto_no_numerico' AS INT) AS ejemplo_falla_controlada
FROM ventas_cast_v
''').show()

### Como interpretar el resultado -- schemas y conversiones

- `printSchema` permite verificar el contrato antes de analizar.
- `cast` transforma tipos; `try_cast` evita que una conversion imposible rompa toda la consulta.
- En pipelines reales, declarar schema reduce errores silenciosos.

In [ ]:
# Schema enforcement en Delta: escribir con contrato controlado
SCHEMA_TABLE = nombre_tabla("sesion9_schema_demo")

spark.sql(f"DROP TABLE IF EXISTS {SCHEMA_TABLE}")
ventas_cast.write.format("delta").mode("overwrite").saveAsTable(SCHEMA_TABLE)

print("Tabla inicial:")
spark.read.table(SCHEMA_TABLE).printSchema()

ventas_extra = ventas_cast.withColumn("canal", F.lit("online"))

try:
    ventas_extra.write.format("delta").mode("append").saveAsTable(SCHEMA_TABLE)
except Exception as exc:
    print("Append con columna extra fallo por schema enforcement.")
    print(f"Detalle: {type(exc).__name__}: {exc}")

print("Para evolucion controlada del schema se usa mergeSchema u operaciones ALTER TABLE.")

In [ ]:
# Pregunta interactiva 6 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 6 de 14 -- Schemas</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> El schema es el contrato del dato.
  </div>
  <p><strong>Por que declarar schema ayuda?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q6" value="A"> A. Evita toda ejecucion</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q6" value="B"> B. Reduce errores de inferencia y cambios silenciosos</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q6" value="C"> C. Convierte todo a texto</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q6" value="D"> D. Elimina permisos</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q6]:checked');
    var out = document.getElementById('fb_q6');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'B') {
      out.innerHTML = 'Correcto. Un contrato explicito mejora confiabilidad.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Un contrato explicito mejora confiabilidad.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q6" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 6 -- Lectura y escritura: CSV, JSON, Parquet y Delta

## Definicion formal

Spark puede leer y escribir multiples formatos. Para una introduccion, los mas
importantes son CSV, JSON, Parquet y Delta.

| Formato | Uso tipico |
|---|---|
| CSV | Intercambio simple, datos pequenos o fuentes legacy |
| JSON | Datos semiestructurados |
| Parquet | Analitica columnar eficiente |
| Delta | Tablas ACID sobre Parquet con historial |

## Modos de escritura

`overwrite` reemplaza, `append` agrega, `ignore` no hace nada si existe,
`error` falla si ya existe.

In [ ]:
# Crear datasets sinteticos para mostrar lectura/escritura sin depender de archivos locales
from pyspark.sql import functions as F

io_base = spark.createDataFrame([
    (1, "Bogota", "2026-01-01", 120.0),
    (2, "Cali", "2026-01-02", 90.5),
    (3, "Medellin", "2026-01-03", 150.2),
], ["id", "ciudad", "fecha_txt", "valor"])

io_base = io_base.withColumn("fecha", F.to_date("fecha_txt")).drop("fecha_txt")
io_base.show()

In [ ]:
# Parquet en Volume si existe permiso; si no, seguir con tabla administrada
VOLUME_NAME = "sesion9_archivos"
BASE_IO_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
PARQUET_PATH = f"{BASE_IO_PATH}/io_demo_parquet"
JSON_PATH = f"{BASE_IO_PATH}/io_demo_json"
CSV_PATH = f"{BASE_IO_PATH}/io_demo_csv"

try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
    io_base.write.mode("overwrite").parquet(PARQUET_PATH)
    io_base.write.mode("overwrite").json(JSON_PATH)
    io_base.write.mode("overwrite").option("header", True).csv(CSV_PATH)

    print("Lectura Parquet:")
    spark.read.parquet(PARQUET_PATH).show()

    print("Lectura JSON:")
    spark.read.json(JSON_PATH).show()

    print("Lectura CSV con opciones:")
    spark.read.option("header", True).option("inferSchema", True).csv(CSV_PATH).show()
except Exception as exc:
    print("No fue posible crear o escribir en un Volume.")
    print("Seguimos con tablas administradas, que funcionan bien para la clase.")
    print(f"Detalle: {type(exc).__name__}: {exc}")
    PARQUET_TABLE = nombre_tabla("sesion9_parquet_demo")
    io_base.write.format("parquet").mode("overwrite").saveAsTable(PARQUET_TABLE)
    print(f"Tabla Parquet administrada creada: {PARQUET_TABLE}")
    spark.read.table(PARQUET_TABLE).show()

### Como interpretar el resultado -- datos locales, Volumes y formatos

- Databricks no lee directamente `C:\Users`; necesita rutas accesibles al workspace.
- Parquet conserva schema y es columnar; CSV necesita opciones e inferencia.
- En serverless, Volumes o tablas administradas son mas seguros que depender de DBFS legacy.

## `saveAsTable()` vs `write.save()`

- `saveAsTable("catalog.schema.tabla")` crea una tabla gobernada.
- `write.save("/Volumes/...")` escribe archivos en un Volume.
- Para analitica repetible, prefiere tablas Delta.

In [ ]:
# Leer, transformar y escribir como tabla Delta
DESTINO_IO = nombre_tabla("sesion9_io_delta")

resultado_io = (
    io_base
    .withColumn("valor_con_iva", F.round(F.col("valor") * 1.19, 2))
    .withColumn("anio", F.year("fecha"))
)

(
    resultado_io.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DESTINO_IO)
)

spark.read.table(DESTINO_IO).show()

In [ ]:
# COPY INTO y Auto Loader: patrones de ingesta
print("COPY INTO para ingesta incremental desde archivos:")
print(f'''
COPY INTO {DESTINO_IO}
FROM '/Volumes/<catalog>/<schema>/<volume>/nuevos_archivos/'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true')
''')

print("Auto Loader para streaming de archivos:")
print('''
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .load("/Volumes/<catalog>/<schema>/<volume>/raw/")
)
''')

In [ ]:
# Pregunta interactiva 7 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 7 de 14 -- Parquet</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Parquet es columnar y Delta agrega log transaccional.
  </div>
  <p><strong>Que afirmacion es correcta?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q7" value="A"> A. Parquet y Delta son identicos</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q7" value="B"> B. Delta usa Parquet mas transaction log</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q7" value="C"> C. CSV siempre es mas eficiente</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q7" value="D"> D. Delta solo sirve para imagenes</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q7]:checked');
    var out = document.getElementById('fb_q7');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'B') {
      out.innerHTML = 'Correcto. Delta agrega ACID, historial y MERGE sobre datos Parquet.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Delta agrega ACID, historial y MERGE sobre datos Parquet.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q7" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 7 -- Lazy evaluation, Catalyst, Jobs, Stages y repartition

## Definicion formal

Spark usa **lazy evaluation**: las transformaciones construyen un plan, pero no
ejecutan trabajo hasta que aparece una accion. **Catalyst** optimiza ese plan.

```
Codigo PySpark -> Logical plan -> Optimized plan -> Physical plan -> Jobs/Stages/Tasks
```

## Intuicion

Cuando escribes `filter`, `select` o `withColumn`, Spark todavia esta planeando.
Cuando escribes `count`, `show`, `collect`, `toPandas` o `write`, Spark ejecuta.

In [ ]:
# Lazy evaluation: construir un plan es rapido porque aun no lee todos los datos
import time
from pyspark.sql import functions as F

t0 = time.perf_counter()
pipeline = (
    sdf
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0.1)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("tarifa_por_milla", F.col("fare_amount") / F.col("trip_distance"))
)
print(f"Construir plan: {(time.perf_counter() - t0) * 1000:.2f} ms")
print(pipeline)

In [ ]:
# explain en varios modos
print("PLAN SIMPLE")
pipeline.explain(False)

print("\nPLAN EXTENDED")
pipeline.explain("extended")

print("\nPLAN FORMATTED")
pipeline.explain("formatted")

### Como interpretar el resultado -- planes de Spark

- `Project` suele indicar seleccion o columnas derivadas.
- `Filter` representa filtros.
- `Exchange` normalmente indica shuffle, una redistribucion costosa.

## Jobs, Stages y Tasks

- Una **accion** dispara normalmente un Job.
- Un **Stage** es una secuencia de operaciones que puede ejecutarse sin shuffle.
- Un **Task** es la unidad de trabajo paralela sobre una particion.
- Cada `Exchange` suele partir el DAG en nuevos stages.

In [ ]:
# Predicate pushdown: seleccionar columnas y filtrar temprano
plan_con_filtro = (
    leer_taxi()
    .select("fare_amount", "trip_distance", "pickup_zip")
    .filter(F.col("fare_amount").between(10, 50))
    .filter(F.col("trip_distance") > 1)
)

plan_con_filtro.explain("formatted")
print(f"Filas resultantes: {plan_con_filtro.count():,}")

In [ ]:
# repartition vs coalesce
pequeno = spark.range(0, 1000)

try:
    print("Particiones iniciales:", pequeno.rdd.getNumPartitions())
    print("repartition(8):", pequeno.repartition(8).rdd.getNumPartitions())
    print("coalesce(1):", pequeno.coalesce(1).rdd.getNumPartitions())
except Exception as exc:
    print("En Spark Connect algunas APIs RDD pueden no estar disponibles.")
    print("Concepto: repartition hace shuffle balanceado; coalesce reduce particiones con menor costo pero puede desbalancear.")
    print(f"Detalle: {exc}")

In [ ]:
# Cache: demo conceptual compatible con entornos donde cache puede estar limitado
base = pipeline.select("pickup_hour", "fare_amount", "tarifa_por_milla")

try:
    base.cache()
    print("Primera accion materializa cache:")
    print(base.count())
    print("Segunda accion puede reutilizar cache:")
    base.groupBy("pickup_hour").count().show(5)
    base.unpersist()
except Exception as exc:
    print("Cache no disponible o limitado en este compute.")
    print("En algunos entornos administrados pueden existir restricciones de cache DataFrame/SQL.")
    print(f"Detalle: {type(exc).__name__}: {exc}")

In [ ]:
# Pregunta interactiva 8 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 8 de 14 -- Lazy evaluation</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Spark no ejecuta transformaciones hasta una accion.
  </div>
  <p><strong>Cual es una accion?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q8" value="A"> A. filter</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q8" value="B"> B. select</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q8" value="C"> C. withColumn</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q8" value="D"> D. count</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q8]:checked');
    var out = document.getElementById('fb_q8');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'D') {
      out.innerHTML = 'Correcto. `count` dispara ejecucion.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. `count` dispara ejecucion.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q8" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 8 -- Photon y Liquid Clustering

## Photon

**Photon** es un motor de ejecucion vectorizado de Databricks. Acelera muchas
consultas SQL/DataFrame sin cambiar el codigo.

## Liquid Clustering

**Liquid Clustering** organiza tablas Delta segun columnas de consulta frecuentes.
Es el reemplazo moderno de muchos patrones basados en `PARTITION BY` y `ZORDER`.

In [ ]:
# Crear tabla Delta con Liquid Clustering
LC_TABLE = nombre_tabla("taxi_liquid_sesion9")

try:
    spark.sql(f'''
    CREATE OR REPLACE TABLE {LC_TABLE}
    CLUSTER BY (tpep_pickup_datetime, fare_amount)
    AS
    SELECT *
    FROM taxi_source_v
    WHERE fare_amount > 0
    ''')
except Exception as exc:
    print("Liquid Clustering no esta disponible en este entorno; creando tabla Delta normal.")
    print(f"Detalle: {type(exc).__name__}: {exc}")
    spark.sql(f'''
    CREATE OR REPLACE TABLE {LC_TABLE}
    USING DELTA
    AS
    SELECT *
    FROM taxi_source_v
    WHERE fare_amount > 0
    ''')

spark.sql(f"DESCRIBE DETAIL {LC_TABLE}").select(
    "format", "clusteringColumns", "numFiles", "sizeInBytes"
).show(truncate=False)

In [ ]:
# OPTIMIZE aplica fisicamente la organizacion
try:
    spark.sql(f"OPTIMIZE {LC_TABLE}")
except Exception as exc:
    print("OPTIMIZE no esta disponible en este entorno o runtime.")
    print(f"Detalle: {type(exc).__name__}: {exc}")

spark.sql(f"DESCRIBE HISTORY {LC_TABLE}").select(
    "version", "timestamp", "operation"
).show(5, truncate=False)

## Predictive Optimization

En workspaces que lo tienen habilitado, Databricks puede ejecutar mantenimiento
como `OPTIMIZE` y `VACUUM` automaticamente segun patrones de uso.

In [ ]:
# Pregunta interactiva 9 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 9 de 14 -- Photon</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Photon acelera consultas compatibles sin cambiar codigo.
  </div>
  <p><strong>Donde se verifica el rendimiento?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q9" value="A"> A. Query Profile</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q9" value="B"> B. Nombre del archivo</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q9" value="C"> C. Ruta C:/Users</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q9" value="D"> D. Markdown</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q9]:checked');
    var out = document.getElementById('fb_q9');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'A') {
      out.innerHTML = 'Correcto. Query Profile muestra detalles de ejecucion.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Query Profile muestra detalles de ejecucion.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q9" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 9 -- Funciones de cadenas, fechas y colecciones

## Definicion formal

`pyspark.sql.functions` contiene funciones nativas que Spark puede optimizar.
Para una primera introduccion, es mejor preferir estas funciones antes que UDFs.

In [ ]:
# Funciones de cadenas
from pyspark.sql import functions as F

texto_df = spark.createDataFrame([
    (1, "  Bogota Norte  ", "factura-2026-0001"),
    (2, "cali sur", "factura-2026-0002"),
    (3, "MEDELLIN centro", "recibo-2025-0099"),
], ["id", "zona", "documento"])

texto_res = (
    texto_df
    .withColumn("zona_limpia", F.trim("zona"))
    .withColumn("zona_upper", F.upper("zona_limpia"))
    .withColumn("largo", F.length("zona_limpia"))
    .withColumn("tipo_doc", F.regexp_extract("documento", r"^([a-z]+)", 1))
    .withColumn("anio_doc", F.regexp_extract("documento", r"(\d{4})", 1))
    .withColumn("zona_partes", F.split(F.lower("zona_limpia"), " "))
    .withColumn("etiqueta", F.concat_ws(" | ", "zona_upper", "documento"))
)
texto_res.show(truncate=False)

In [ ]:
# Funciones de fechas y tiempo
fechas_df = spark.createDataFrame([
    ("2026-01-05 08:30:00",),
    ("2026-02-10 14:45:00",),
    ("2026-03-20 23:05:00",),
], ["ts_txt"])

fechas_res = (
    fechas_df
    .withColumn("ts", F.to_timestamp("ts_txt"))
    .withColumn("fecha", F.to_date("ts"))
    .withColumn("anio", F.year("ts"))
    .withColumn("mes", F.month("ts"))
    .withColumn("dia", F.dayofmonth("ts"))
    .withColumn("hora", F.hour("ts"))
    .withColumn("fecha_mas_7", F.date_add("fecha", 7))
    .withColumn("inicio_mes", F.date_trunc("month", "ts"))
    .withColumn("dias_desde_hoy", F.datediff(F.current_date(), F.col("fecha")))
)
fechas_res.show(truncate=False)

In [ ]:
# Arrays y maps
colecciones = spark.createDataFrame([
    (1, ["spark", "delta", "spark"], {"nivel": "intro", "motor": "spark"}),
    (2, ["sql", "parquet"], {"nivel": "intro", "motor": "sql"}),
], ["id", "temas", "meta"])

colecciones_res = (
    colecciones
    .withColumn("n_temas", F.size("temas"))
    .withColumn("temas_unicos", F.array_distinct("temas"))
    .withColumn("incluye_spark", F.array_contains("temas", "spark"))
    .withColumn("meta_keys", F.map_keys("meta"))
    .withColumn("meta_values", F.map_values("meta"))
)
colecciones_res.show(truncate=False)

colecciones_res.select("id", F.explode("temas_unicos").alias("tema")).show()

In [ ]:
# Operaciones de conjuntos entre DataFrames
a = spark.createDataFrame([(1, "A"), (2, "B"), (3, "C")], ["id", "letra"])
b = spark.createDataFrame([(3, "C"), (4, "D"), (5, "E")], ["id", "letra"])

print("unionByName")
a.unionByName(b).show()

print("intersect")
a.intersect(b).show()

print("subtract")
a.subtract(b).show()

print("distinct despues de union")
a.unionByName(b).distinct().show()

### Como interpretar el resultado -- funciones nativas

- Las funciones nativas permanecen dentro del plan de Spark.
- Spark puede optimizar filtros, proyecciones y expresiones mejor que una UDF Python.
- Estas funciones cubren gran parte del trabajo cotidiano de limpieza.

In [ ]:
# Pregunta interactiva 10 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 10 de 14 -- Funciones</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Las funciones nativas son optimizables.
  </div>
  <p><strong>Que conviene preferir?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q10" value="A"> A. UDF Python siempre</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q10" value="B"> B. Funciones nativas de PySpark</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q10" value="C"> C. collect y for</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q10" value="D"> D. Pandas para todo</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q10]:checked');
    var out = document.getElementById('fb_q10');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'B') {
      out.innerHTML = 'Correcto. Las funciones nativas permanecen dentro del motor Spark.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Las funciones nativas permanecen dentro del motor Spark.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q10" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 10 -- Transformaciones completas de la API PySpark

## Mapa mental

| Tipo | Operaciones |
|---|---|
| Narrow | `select`, `filter`, `withColumn`, `drop` |
| Wide | `groupBy`, `join`, `distinct`, `orderBy` |
| Analiticas | `Window`, `pivot`, percentiles |
| Calidad | `na.drop`, `na.fill`, `dropDuplicates` |

In [ ]:
# Base enriquecida para el tour PySpark
from pyspark.sql.window import Window

enriquecido = (
    sdf
    .select(
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "fare_amount", "trip_distance", "pickup_zip", "dropoff_zip"
    )
    .filter(F.col("fare_amount").between(1, 200))
    .filter(F.col("trip_distance") > 0)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn(
        "duracion_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    .withColumn("tarifa_por_milla", F.col("fare_amount") / F.col("trip_distance"))
    .withColumn(
        "categoria_viaje",
        F.when(F.col("trip_distance") < 1, "micro")
         .when(F.col("trip_distance") < 3, "corto")
         .when(F.col("trip_distance") < 10, "medio")
         .otherwise("largo")
    )
    .filter(F.col("duracion_min").between(1, 180))
)
enriquecido.show(5, truncate=False)

In [ ]:
# groupBy + agg
metricas = (
    enriquecido
    .groupBy("pickup_hour", "categoria_viaje")
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("tarifa_prom"),
        F.round(F.avg("tarifa_por_milla"), 2).alias("tarifa_por_milla_prom"),
        F.round(F.stddev("fare_amount"), 2).alias("tarifa_std"),
        F.round(F.percentile_approx("fare_amount", 0.9), 2).alias("tarifa_p90"),
    )
    .orderBy("pickup_hour", "categoria_viaje")
)
metricas.show(10, truncate=False)
metricas.explain("formatted")

In [ ]:
# Window functions
w_hora = Window.partitionBy("pickup_hour").orderBy(F.desc("viajes"))

top_hora = (
    metricas
    .withColumn("rank_en_hora", F.rank().over(w_hora))
    .filter(F.col("rank_en_hora") <= 2)
    .orderBy("pickup_hour", "rank_en_hora")
)
top_hora.show(20, truncate=False)

In [ ]:
# Join con broadcast
zip_ref = (
    enriquecido.select("pickup_zip")
    .where(F.col("pickup_zip").isNotNull())
    .distinct()
    .limit(500)
    .withColumn(
        "zona",
        F.when(F.col("pickup_zip").between(10001, 10099), "Manhattan")
         .otherwise("Otra")
    )
)

joined = (
    enriquecido
    .join(F.broadcast(zip_ref), on="pickup_zip", how="left")
    .groupBy("zona")
    .agg(F.count("*").alias("viajes"), F.round(F.avg("fare_amount"), 2).alias("tarifa_prom"))
)
joined.show()
joined.explain("formatted")

In [ ]:
# Pivot
pivot_categoria = (
    enriquecido
    .groupBy("pickup_hour")
    .pivot("categoria_viaje", ["micro", "corto", "medio", "largo"])
    .agg(F.count("*"))
    .orderBy("pickup_hour")
)
pivot_categoria.show()

In [ ]:
# Calidad de datos
sdf.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100, 2).alias(c)
    for c in ["fare_amount", "trip_distance", "pickup_zip", "dropoff_zip"]
]).show(truncate=False)

limpio = (
    sdf.na.drop(subset=["fare_amount", "trip_distance"])
       .filter(F.col("fare_amount") > 0)
       .filter(F.col("trip_distance") > 0)
       .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount"])
)
print(f"Filas limpias: {limpio.count():,}")

In [ ]:
# UDF vs pandas_udf vs funcion nativa: patron pedagogico
print("Orden recomendado:")
print("1. Funcion nativa de pyspark.sql.functions")
print("2. pandas_udf si la logica vectorizada en Python es inevitable")
print("3. udf clasica solo cuando no haya alternativa")

clasificacion_nativa = (
    enriquecido
    .withColumn(
        "tipo_duracion",
        F.when(F.col("duracion_min") < 5, "rapido")
         .when(F.col("duracion_min") < 20, "normal")
         .otherwise("largo")
    )
    .groupBy("tipo_duracion")
    .count()
)
clasificacion_nativa.show()

### Como interpretar el resultado -- API PySpark

- La API DataFrame permite escribir transformaciones legibles y optimizables.
- Los shuffles aparecen en agregaciones, joins y pivots.
- Despues de cada salida, interpreta patron descriptivo y limitaciones.

---
# Sección 11 -- Por que Spark sobre Pandas, y cuando no

## Idea clave

Pandas no es "malo" y Spark no es "siempre mejor". Pandas gana cuando el dataset
cabe comodamente en memoria y se necesita iterar rapido. Spark gana cuando el
volumen crece, se requieren pipelines reproducibles, SQL distribuido, observabilidad
y tablas gobernadas.

In [ ]:
# Comparacion representativa: Spark vs Pandas
import time
import pandas as pd

MUESTRA = leer_taxi().limit(500000)

t0 = time.perf_counter()
spark_res = (
    MUESTRA
    .filter(F.col("fare_amount") > 0)
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .groupBy("hora")
    .agg(F.count("*").alias("viajes"), F.round(F.avg("fare_amount"), 2).alias("tarifa_prom"))
    .orderBy("hora")
)
spark_res.show(5)
t_spark = time.perf_counter() - t0

t0 = time.perf_counter()
pdf = MUESTRA.select("fare_amount", "tpep_pickup_datetime").toPandas()
pdf = pdf[pdf["fare_amount"] > 0].copy()
pdf["hora"] = pd.to_datetime(pdf["tpep_pickup_datetime"]).dt.hour
pdf_res = pdf.groupby("hora")["fare_amount"].agg(["count", "mean"]).sort_index()
print(pdf_res.head())
t_pandas = time.perf_counter() - t0

print(f"Spark : {t_spark:.2f}s")
print(f"Pandas: {t_pandas:.2f}s")
print("Nota: el tiempo Pandas incluye toPandas(), que mueve datos al driver.")

## Tabla de decision

| Criterio | Elige Pandas | Elige Spark |
|---|---|---|
| Tamano | Cabe en RAM | Puede superar la RAM |
| Iteracion | Muy rapida | Pipeline estable |
| SQL distribuido | No necesario | Necesario |
| Observabilidad | Baja prioridad | Query Profile / Jobs |
| Tablas Delta | No nativo | Integrado |

In [ ]:
# Pregunta interactiva 11 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 11 de 14 -- Spark vs Pandas</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Pandas es excelente si todo cabe en RAM.
  </div>
  <p><strong>Cuando suele ganar Spark?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q11" value="A"> A. Datos grandes y pipelines reproducibles</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q11" value="B"> B. Cinco filas locales</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q11" value="C"> C. Editar a mano</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q11" value="D"> D. Sin SQL ni crecimiento</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q11]:checked');
    var out = document.getElementById('fb_q11');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'A') {
      out.innerHTML = 'Correcto. Spark gana por escala, SQL distribuido y operacion.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Spark gana por escala, SQL distribuido y operacion.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q11" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 12 -- Por que Spark sobre Dask, y cuando no

## Diferencia arquitectural

Dask escala Python y se integra muy bien con numpy, scipy y scikit-learn. Spark
trabaja con un optimizador SQL/DataFrame maduro: Catalyst. Por eso Spark suele
ser mas fuerte en data engineering, joins grandes, SQL distribuido y lakehouse.

In [ ]:
# Dask vs Spark: ejemplo pequeno sobre la misma muestra
import dask.dataframe as dd

pdf_base = (
    leer_taxi()
    .select("fare_amount", "tpep_pickup_datetime", "pickup_zip")
    .limit(100000)
    .toPandas()
)
pdf_base = pdf_base[pdf_base["fare_amount"] > 0].copy()

ddf = dd.from_pandas(pdf_base, npartitions=8)
sdf_bench = spark.createDataFrame(pdf_base)

t0 = time.perf_counter()
dask_res = (
    ddf.assign(hora=dd.to_datetime(ddf["tpep_pickup_datetime"]).dt.hour)
       .groupby("hora")["fare_amount"]
       .agg(["count", "mean"])
       .compute()
)
t_dask = time.perf_counter() - t0

t0 = time.perf_counter()
spark_bench = (
    sdf_bench
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .groupBy("hora")
    .agg(F.count("*").alias("count"), F.avg("fare_amount").alias("mean"))
)
spark_bench.show(5)
t_spark = time.perf_counter() - t0

print(f"Dask : {t_dask:.2f}s")
print(f"Spark: {t_spark:.2f}s")

In [ ]:
# Join con broadcast en Spark
ref_pdf = pdf_base[["pickup_zip"]].dropna().drop_duplicates().head(100).copy()
ref_pdf["zona"] = "referencia"

ref_sdf = spark.createDataFrame(ref_pdf)
join_spark = sdf_bench.join(F.broadcast(ref_sdf), on="pickup_zip", how="inner")
print(f"Filas join Spark: {join_spark.count():,}")
join_spark.explain("formatted")

## Tabla de decision

| Criterio | Elige Dask | Elige Spark |
|---|---|---|
| Ecosistema numpy/scipy | Prioritario | Secundario |
| Migracion desde Pandas | Gradual | Requiere nueva mentalidad |
| SQL distribuido | Limitado | Nativo |
| Optimizacion de joins | Menor | Catalyst |
| Lakehouse/Delta | No nativo | Integrado |

In [ ]:
# Pregunta interactiva 12 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 12 de 14 -- Spark vs Dask</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Dask escala Python; Spark optimiza planes SQL/DataFrame.
  </div>
  <p><strong>Que ventaja es clara de Spark?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q12" value="A"> A. Catalyst y SQL distribuido</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q12" value="B"> B. Editar Excel</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q12" value="C"> C. No usar tablas</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q12" value="D"> D. Solo numpy local</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q12]:checked');
    var out = document.getElementById('fb_q12');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'A') {
      out.innerHTML = 'Correcto. Catalyst optimiza consultas antes de ejecutarlas.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. Catalyst optimiza consultas antes de ejecutarlas.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q12" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 13 -- Delta Lake avanzado

## Definicion formal

**Delta Lake** guarda datos en archivos Parquet y agrega un transaction log
`_delta_log`. Ese log permite ACID, historial, MERGE, time travel, schema
enforcement y schema evolution.

## Parquet vs Delta

Parquet es formato de archivo. Delta es una capa transaccional sobre Parquet.

In [ ]:
# Crear tabla Delta base
DELTA_MAIN = nombre_tabla("taxi_sesion9_main")

try:
    spark.sql(f'''
    CREATE OR REPLACE TABLE {DELTA_MAIN}
    CLUSTER BY (tpep_pickup_datetime, pickup_zip)
    AS
    SELECT
      CAST(row_number() OVER (ORDER BY tpep_pickup_datetime) AS BIGINT) AS trip_id,
      tpep_pickup_datetime,
      tpep_dropoff_datetime,
      pickup_zip,
      dropoff_zip,
      fare_amount,
      trip_distance,
      CAST(1 AS INT) AS es_valido
    FROM taxi_source_v
    WHERE fare_amount > 0 AND trip_distance > 0
    ''')
except Exception as exc:
    print("CLUSTER BY no disponible; creando tabla Delta normal.")
    print(f"Detalle: {type(exc).__name__}: {exc}")
    spark.sql(f'''
    CREATE OR REPLACE TABLE {DELTA_MAIN}
    USING DELTA
    AS
    SELECT
      CAST(row_number() OVER (ORDER BY tpep_pickup_datetime) AS BIGINT) AS trip_id,
      tpep_pickup_datetime,
      tpep_dropoff_datetime,
      pickup_zip,
      dropoff_zip,
      fare_amount,
      trip_distance,
      CAST(1 AS INT) AS es_valido
    FROM taxi_source_v
    WHERE fare_amount > 0 AND trip_distance > 0
    ''')

spark.sql(f"DESCRIBE DETAIL {DELTA_MAIN}").select("format", "numFiles", "sizeInBytes").show()

In [ ]:
# MERGE: upsert
from delta.tables import DeltaTable

updates = spark.createDataFrame([
    (1, 0),
    (2, 0),
    (999999999, 1),
], ["trip_id", "es_valido"])

target = DeltaTable.forName(spark, DELTA_MAIN)

(
    target.alias("t")
    .merge(updates.alias("s"), "t.trip_id = s.trip_id")
    .whenMatchedUpdate(set={"es_valido": "s.es_valido"})
    .whenNotMatchedInsert(values={
        "trip_id": "s.trip_id",
        "tpep_pickup_datetime": "CAST(NULL AS TIMESTAMP)",
        "tpep_dropoff_datetime": "CAST(NULL AS TIMESTAMP)",
        "pickup_zip": "CAST(NULL AS INT)",
        "dropoff_zip": "CAST(NULL AS INT)",
        "fare_amount": "CAST(0 AS DOUBLE)",
        "trip_distance": "CAST(0 AS DOUBLE)",
        "es_valido": "s.es_valido",
    })
    .execute()
)

spark.sql(f"DESCRIBE HISTORY {DELTA_MAIN}").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(5, truncate=False)

In [ ]:
# Time Travel y RESTORE
version_0 = spark.read.format("delta").option("versionAsOf", 0).table(DELTA_MAIN).count()
actual = spark.read.table(DELTA_MAIN).count()

print(f"Version 0: {version_0:,}")
print(f"Actual   : {actual:,}")

spark.sql(f"RESTORE TABLE {DELTA_MAIN} TO VERSION AS OF 0")
spark.sql(f"DESCRIBE HISTORY {DELTA_MAIN}").select("version", "timestamp", "operation").show(5, truncate=False)

## Schema evolution, CONVERT y CLONE

- **Schema enforcement** evita escribir columnas inesperadas.
- **Schema evolution** permite agregar columnas de forma controlada.
- **CONVERT TO DELTA** convierte Parquet existente a Delta.
- **SHALLOW CLONE** copia metadatos y apunta a los mismos archivos.
- **DEEP CLONE** copia tambien archivos fisicos.

In [ ]:
# Schema evolution con ALTER TABLE y mergeSchema
spark.sql(f"ALTER TABLE {DELTA_MAIN} ADD COLUMNS (comentario_calidad STRING)")

df_nueva_col = spark.read.table(DELTA_MAIN).limit(10).withColumn("fuente_lote", F.lit("demo"))
(
    df_nueva_col.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DELTA_MAIN)
)

spark.read.table(DELTA_MAIN).printSchema()

In [ ]:
# CLONE: crear tabla de prueba
CLONE_TABLE = nombre_tabla("taxi_sesion9_clone")

try:
    spark.sql(f"DROP TABLE IF EXISTS {CLONE_TABLE}")
    spark.sql(f"CREATE TABLE {CLONE_TABLE} SHALLOW CLONE {DELTA_MAIN}")
    spark.sql(f"DESCRIBE HISTORY {CLONE_TABLE}").select("version", "timestamp", "operation").show(5, truncate=False)
except Exception as exc:
    print("CLONE puede no estar disponible en este workspace.")
    print(f"Detalle: {type(exc).__name__}: {exc}")

In [ ]:
# VACUUM: eliminar archivos obsoletos
print("VACUUM debe usarse con cuidado porque limita time travel a versiones antiguas.")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql(f"VACUUM {DELTA_MAIN} RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")
spark.sql(f"DESCRIBE DETAIL {DELTA_MAIN}").select("numFiles", "sizeInBytes").show()

In [ ]:
# Pregunta interactiva 13 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 13 de 14 -- Delta Lake</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Delta tiene transaction log.
  </div>
  <p><strong>Que habilita?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q13" value="A"> A. MERGE, time travel y ACID</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q13" value="B"> B. Leer disco local C:</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q13" value="C"> C. Eliminar schemas</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q13" value="D"> D. Evitar todos los jobs</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q13]:checked');
    var out = document.getElementById('fb_q13');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'A') {
      out.innerHTML = 'Correcto. El log permite control transaccional e historial.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. El log permite control transaccional e historial.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q13" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 14 -- Lakeflow / Delta Live Tables

## Definicion formal

**Lakeflow Spark Declarative Pipelines** es la evolucion del producto conocido
como **Delta Live Tables (DLT)**. La API Python todavia usa el modulo `dlt`.

No se ejecuta como celda interactiva comun: se configura como pipeline. Esta
seccion imprime el patron para que el estudiante entienda la arquitectura.

In [ ]:
# Codigo pedagogico de pipeline Lakeflow/DLT
PIPELINE_CODE = '''
import dlt
from pyspark.sql import functions as F

@dlt.view(name="taxi_raw_view")
def taxi_raw_view():
    return spark.read.table("default.taxi_source_delta")

@dlt.table(name="taxi_bronze", comment="Ingesta raw")
def taxi_bronze():
    return dlt.read("taxi_raw_view")

@dlt.table(name="taxi_silver", comment="Datos limpios")
@dlt.expect_all({
    "fare_positivo": "fare_amount > 0",
    "distancia_positiva": "trip_distance > 0"
})
@dlt.expect_or_drop("duracion_valida", "tpep_dropoff_datetime >= tpep_pickup_datetime")
def taxi_silver():
    return (
        dlt.read("taxi_bronze")
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn("tarifa_por_milla", F.col("fare_amount") / F.col("trip_distance"))
    )

@dlt.table(name="taxi_gold_hourly", comment="Metricas por hora")
def taxi_gold_hourly():
    return (
        dlt.read("taxi_silver")
        .groupBy("pickup_hour")
        .agg(
            F.count("*").alias("viajes"),
            F.round(F.avg("fare_amount"), 2).alias("tarifa_prom")
        )
    )
'''

print(PIPELINE_CODE)
print("Para ejecutar: Workflows -> Lakeflow Declarative Pipelines -> Create pipeline")

## Batch vs streaming

- Usa **batch** cuando reprocesas lotes completos o tablas estables.
- Usa **streaming** cuando llegan archivos o eventos nuevos continuamente.
- En Databricks Free/Community serverless esta seccion es introductoria; verifica los triggers soportados por tu workspace.

## Parametros

Un pipeline puede leer parametros con `spark.conf.get("pipeline.parametro")`.
Esto permite cambiar fuentes, fechas o modos sin editar codigo.

In [ ]:
# Patron streaming pedagogico: imprimir, no ejecutar aqui
STREAMING_PATTERN = '''
import dlt

@dlt.table(name="eventos_bronze_stream")
def eventos_bronze_stream():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/Volumes/<catalog>/<schema>/<volume>/raw_events/")
    )
'''

print(STREAMING_PATTERN)

---
# Sección 15 -- Databricks Workflows y Jobs

## Definicion formal

Un **Job** ejecuta una tarea de forma reproducible. Un **Workflow** puede contener
varias tareas conectadas como DAG: notebooks, Python scripts, SQL, pipelines
Lakeflow, dbt u otros tipos.

## Intuicion

El notebook interactivo sirve para aprender y explorar. El Job sirve para operar:
programar, parametrizar, monitorear, reintentar y notificar.

## Conceptos clave

| Concepto | Explicacion |
|---|---|
| Task | Unidad ejecutable dentro de un Job |
| Job cluster | Compute creado para el Job |
| Existing compute | Compute reutilizado |
| Schedule | Programacion cron o trigger |
| Parameters | Valores que cambian sin editar codigo |
| Notifications | Alertas por exito, falla o duracion |

## Jobs vs Lakeflow

Usa **Jobs** para orquestacion general: notebooks, SQL, scripts, modelos, reportes.
Usa **Lakeflow** cuando el problema central es declarar tablas de datos con
dependencias, calidad y procesamiento incremental.

In [ ]:
# Patron para que un notebook sea invocable como task
dbutils.widgets.text("fecha_proceso", "2026-01-01", "Fecha de proceso")
dbutils.widgets.dropdown("modo", "demo", ["demo", "produccion"], "Modo")

fecha_proceso = dbutils.widgets.get("fecha_proceso")
modo = dbutils.widgets.get("modo")

print(f"Ejecutando notebook con fecha_proceso={fecha_proceso}, modo={modo}")

# En un Job real, al final se puede devolver un resultado textual:
# dbutils.notebook.exit("ok")

In [ ]:
# Pregunta interactiva 14 de 14
# Estilo IRdisplay adaptado a Databricks/Python: caja HTML con displayHTML.
html = '''
<div style="border:2px solid #2563eb; background:#eff6ff; border-radius:8px; padding:16px; margin:12px 0; font-family:Arial, sans-serif;">
  <h3 style="margin:0 0 10px 0; color:#1d4ed8;">Pregunta 14 de 14 -- Workflows</h3>
  <div style="background:#fef3c7; border-left:5px solid #f59e0b; padding:10px; margin:10px 0;">
    <strong>Contexto.</strong> Un Job operacionaliza un notebook.
  </div>
  <p><strong>Que pregunta resume la sesion?</strong></p>
  <label style="display:block; margin:8px 0;"><input type="radio" name="q14" value="A"> A. Como traigo todo al driver?</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q14" value="B"> B. Donde vive el dato, que plan ejecuta Spark y como lo opero?</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q14" value="C"> C. Como evito tablas?</label>
<label style="display:block; margin:8px 0;"><input type="radio" name="q14" value="D"> D. Como reemplazo Spark con for loops?</label>
  <button onclick="
    var marcado = document.querySelector('input[name=q14]:checked');
    var out = document.getElementById('fb_q14');
    if (!marcado) {
      out.innerHTML = 'Selecciona una opcion antes de verificar.';
      out.style.background = '#fef3c7';
      out.style.color = '#92400e';
      return;
    }
    if (marcado.value === 'B') {
      out.innerHTML = 'Correcto. La mentalidad correcta conecta datos, motor, tablas y operacion.';
      out.style.background = '#dcfce7';
      out.style.color = '#166534';
    } else {
      out.innerHTML = 'Incorrecto. La mentalidad correcta conecta datos, motor, tablas y operacion.';
      out.style.background = '#fee2e2';
      out.style.color = '#991b1b';
    }
  " style="background:#2563eb; color:white; border:0; border-radius:6px; padding:8px 12px; cursor:pointer;">
    Verificar respuesta
  </button>
  <div id="fb_q14" style="margin-top:10px; padding:10px; border-radius:6px;"></div>
</div>
'''

try:
    displayHTML(html)
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))

---
# Sección 16 -- Taller end-to-end

## Objetivo del taller

Aplicar los conceptos de la sesion en ejercicios guiados. Cada ejercicio tiene
instrucciones y deja un `NotImplementedError` para que el estudiante complete.

In [ ]:
# Ejercicio 1 -- Window functions
# Construye el top 3 de categorias de viaje por hora con mayor tarifa_por_milla promedio.
# Requisitos:
# - Leer la fuente de taxis preparada en `leer_taxi()`.
# - Crear pickup_hour, tarifa_por_milla y categoria_viaje.
# - Agrupar por pickup_hour y categoria_viaje.
# - Filtrar grupos con menos de 100 viajes.
# - Usar Window.partitionBy("pickup_hour").orderBy(F.desc("tarifa_por_milla_prom")).

raise NotImplementedError("Completa el ejercicio 1 siguiendo las instrucciones.")

In [ ]:
# Ejercicio 2 -- MERGE en Delta
# Crea una tabla Delta con viajes del pickup_zip mas frecuente.
# Luego usa MERGE para marcar es_valido=0 donde fare_amount > 100 e insertar 3 filas nuevas.

raise NotImplementedError("Completa el ejercicio 2 siguiendo las instrucciones.")

In [ ]:
# Ejercicio 3 -- Reporte de calidad
# Construye un DataFrame [metrica, valor] con:
# - pct_nulos por columna
# - pct_negativos para columnas numericas
# - top 5 pickup_zip
# - total_filas

raise NotImplementedError("Completa el ejercicio 3 siguiendo las instrucciones.")

In [ ]:
# Ejercicio 4 -- Schema + I/O
# Define un StructType de 5 columnas, crea DataFrame sintetico, escribe con saveAsTable,
# lee de vuelta, verifica schema y agrega una columna con ALTER TABLE.

raise NotImplementedError("Completa el ejercicio 4 siguiendo las instrucciones.")

In [ ]:
# Ejercicio 5 -- Pipeline completo Databricks
# Lee la fuente de taxis preparada, filtra, enriquece con 5 columnas derivadas,
# crea tabla Silver con Liquid Clustering, ejecuta MERGE con 5 actualizaciones
# y muestra DESCRIBE HISTORY.

raise NotImplementedError("Completa el ejercicio 5 siguiendo las instrucciones.")

## Checklist final

```
[ ] Uso `catalog.schema.table` cuando Unity Catalog esta disponible
[ ] Entiendo por que C:\Users no funciona dentro de Databricks
[ ] Uso Volumes o tablas administradas en lugar de depender de DBFS legacy
[ ] Prefiero DataFrames/Spark SQL sobre RDDs para el trabajo principal
[ ] Uso %pip, no %sh pip
[ ] Puedo leer CSV, JSON, Parquet y Delta
[ ] Puedo explicar Parquet vs Delta
[ ] Puedo leer un plan con explain()
[ ] Reconozco Exchange como posible shuffle
[ ] Priorizo funciones nativas sobre UDFs
[ ] Entiendo el patron bronze/silver/gold
[ ] Se cuando usar Jobs y cuando Lakeflow
```

## Cierre

La idea mas importante: Databricks no es solo un notebook. Es una plataforma
para convertir datos en tablas confiables, transformaciones reproducibles y
ejecuciones gobernadas.

## Referencias

- Databricks Free Edition: https://docs.databricks.com/aws/en/getting-started/free-edition
- Databricks Free Edition limitations: https://docs.databricks.com/aws/en/getting-started/free-edition-limitations
- Serverless compute limitations: https://docs.databricks.com/aws/en/compute/serverless/limitations
- Databricks notebooks: https://docs.databricks.com/en/notebooks/
- DBFS: https://docs.databricks.com/en/dbfs/
- Databricks widgets: https://docs.databricks.com/en/notebooks/widgets.html
- Unity Catalog Volumes: https://docs.databricks.com/aws/en/volumes/
- Apache Spark documentation: https://spark.apache.org/docs/latest/
- PySpark functions: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html
- Apache Parquet: https://parquet.apache.org/docs/
- Delta Lake: https://docs.delta.io/latest/index.html
- Lakeflow Declarative Pipelines: https://docs.databricks.com/en/delta-live-tables/index.html